# mart_dev.job_costs

A Saved table made by the batch costs, with project.convert(...), from the statements in `conversion/statement.hql`, as its script wrote them. Its logic cell builds the same query from the Toolbox's own functions, with create(), preview() and write(). Answer its open Decisions, which `conversion/checklist.md` lists, in the decisions cell of `admin/workflows/convert/convert.ipynb`, then run the cells top to bottom. The cells comparing it with the original stay until you delete them.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project: two up

from sqlglot_composer import (compare_hive, compare_with_original, export_hive, export_lineage,
                              project, run_query, show_hive)
from warehouse import query

In [ ]:
# logic
from sqlglot_composer import (
    Statement,
    unchecked_hive,
)
from sqlglot_composer import create_table  # for create()
from sqlglot_composer import UncheckedStatement, unchecked_hive  # for write()
from references.mart_dev__job_costs import job_costs  # the Saved table's own Table reference


def converted() -> Statement:
    """The query, as the Hive this Conversion was made from writes it."""
    return unchecked_hive("WITH job_runs AS (SELECT job_id, duration_mins FROM ops.job_runs) SELECT c.job_id, CASE WHEN c.job_id = 3 THEN 1000 ELSE COALESCE(m.total_minutes, 0) * 50 END AS cost_cents FROM (SELECT 1 AS job_id UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4) AS c LEFT JOIN (SELECT job_id, SUM(duration_mins) AS total_minutes FROM job_runs GROUP BY job_id) AS m ON c.job_id = m.job_id")


def create() -> Statement:
    """Create mart_dev.job_costs from its Table reference, if it isn't there yet."""
    return create_table(job_costs, may_exist=True)


def preview() -> Statement:
    """The rows write() would save, as a SELECT to run and check first."""
    return converted()


def write() -> UncheckedStatement:
    """Save the rows into mart_dev.job_costs, as the original's INSERT did: kept as Hive,
    as converted() is."""
    return unchecked_hive("WITH job_runs AS (SELECT job_id, duration_mins FROM ops.job_runs) INSERT OVERWRITE TABLE mart_dev.job_costs SELECT c.job_id, CASE WHEN c.job_id = 3 THEN 1000 ELSE COALESCE(m.total_minutes, 0) * 50 END AS cost_cents FROM (SELECT 1 AS job_id UNION ALL SELECT 2 UNION ALL SELECT 3 UNION ALL SELECT 4) AS c LEFT JOIN (SELECT job_id, SUM(duration_mins) AS total_minutes FROM job_runs GROUP BY job_id) AS m ON c.job_id = m.job_id")


In [ ]:
# Try it: its rows, then, once its Table reference has its location=, the Hive of
# each step
run_query(preview(), query)
show_hive([create(), write()])

In [ ]:
# Lineage: a new pair in lineage/, named by the time and this table
export_lineage(write())

In [ ]:
# Its Hive: mart_dev__job_costs.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# Its Hive against the original's, as conversion/in_dev.hql holds it
ORIGINAL = pathlib.Path("conversion/in_dev.hql").read_text(encoding="utf-8")
show_hive(preview())
print(compare_hive(preview(), ORIGINAL))

In [ ]:
# The convert Workflow's compare step, project.compare_batch, compares this table once
# written. This compares its query with the original's first:
# Its rows against the original's, through your query, on one Partition:
# print(compare_with_original(preview(), ORIGINAL, query=query, partition={"dt": "<a value>"}))

In [ ]:
# Its rows in the Reference sheet
project.refresh_reference_sheet("mart_dev.job_costs")